GPT-2 Model Configuration

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "stride": 128,
    "embedding_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "dropout": 0.1,
    "qkv_bias": False
}

Load Dataset

In [3]:

file_path = "../the-verdict.txt"

with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

Tokenization

In [4]:
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

In [5]:
token_ids = tokenizer.encode(text_data)

In [6]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

In [7]:
len(token_ids)

5145

Implementing Dataloader

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_len, stride):

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_len, stride):
            input_chunk = token_ids[i:i+max_len]
            target_chunk = token_ids[i+1:i+max_len+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    
def create_dataloader(text, tokenizer, batch_size, context_len, stride, shuffle, drop_last, num_workers):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(

        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers    
    )

    return dataloader

In [9]:
train_ratio = 0.8

train_data = text_data[:int(train_ratio*len(text_data))]
test_data = text_data[int(train_ratio*len(text_data)):]

torch.manual_seed(123)

train_dataloader = create_dataloader(

    text=train_data,
    tokenizer=tokenizer,
    batch_size=2,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=True,
    drop_last=True,
    num_workers=0

)

test_dataloader = create_dataloader(
    
    text=test_data,
    tokenizer=tokenizer,
    batch_size=2,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=False,
    drop_last=False,
    num_workers=0

)

In [10]:
for x, y in train_dataloader:
    print(x.shape, y.shape)

for x, y in test_dataloader:
    print(x.shape, y.shape)


torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([1, 256]) torch.Size([1, 256])


Multi-Head Attention

In [11]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, num_heads, context_len, dropout, qkv_bias):
        super().__init__()

        assert(d_out % num_heads ==0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = (d_out // num_heads)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # print("Keys:")
        # print(keys)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # print(queries)
        
        # print(values)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_score = queries @ keys.transpose(2, 3)
        # print(attn_score)
        
        masked_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_score.masked_fill_(masked_bool, -torch.inf)
        # print(attn_score)

        attn_weights = torch.softmax(attn_score / (keys.shape[-1] ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)
        # print(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2)

        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        # print(context_vector)

        return context_vector
        


Layer Normalization, GELU Activation Function and Feed Forward Neural Network

In [12]:
class LayerNormaliztion(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        x_norm = (x - mean)/torch.sqrt(var + self.eps)
        # print(x_norm)

        return self.scale * x_norm + self.shift


class GELUActivation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))
    
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["embedding_dim"], 4*cfg["embedding_dim"]),
            GELUActivation(),
            nn.Linear(4*cfg["embedding_dim"], cfg["embedding_dim"])

        )
        
    def forward(self, x):
        return self.layers(x)

Transformer Block

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["embedding_dim"],
            d_out=cfg["embedding_dim"],
            num_heads=cfg["num_heads"],
            context_len=cfg["context_length"],
            dropout=cfg["dropout"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.norm1 = LayerNormaliztion(cfg["embedding_dim"])
        self.norm2 = LayerNormaliztion(cfg["embedding_dim"])
        self.ff = FeedForward(cfg)
        self.dropout = nn.Dropout(cfg["dropout"])
    
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut
        # print(x)

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

GPT Model Architecture

In [14]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout"])

        self.transformers = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        self.final_norm = LayerNormaliztion(cfg["embedding_dim"])
        self.final_output = nn.Linear(cfg["embedding_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.dropout(x)
        # print("Token Embeddings")
        # print(tok_emb)

        x = self.transformers(x)
        # print(x)

        x = self.final_norm(x)
        logits = self.final_output(x)

        return logits
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

Loss Function

In [15]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    # print(logits)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    # print(loss)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [16]:
def evaluate_model(model, train_loader, test_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(test_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

Training Loop for LLM

In [17]:
def train_model(model, train_loader, test_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    train_losses, test_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch + 1}")
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen = tokens_seen + input_batch.numel()
            global_step += 1

            if(global_step % eval_freq ==0):
                train_loss, test_loss = evaluate_model(model, train_loader, test_loader, device, eval_iter)

                train_losses.append(train_loss)
                test_losses.append(test_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Test loss {test_loss:.3f}")
                
        
        
    
    return train_losses, test_losses, track_tokens_seen         


In [18]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [19]:
print(torch.cuda.memory_allocated()/1024**3, "GB allocated")
print(torch.cuda.memory_reserved()/1024**3, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [20]:
import time
start_time = time.time()

gpu = torch.device('cuda')
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)



model.to(gpu)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0004, weight_decay=0.1)

num_epochs = 10

train_losses, test_losses, tokens_seen = train_model(
    model, train_dataloader, test_dataloader, optimizer, gpu, num_epochs, eval_freq=5, eval_iter=5
)

end_time = time.time()

time_taken = (end_time - start_time)/60

print(f"Time Taken for Training in minutes: {time_taken} Minutes")

Epoch: 1
Ep 1 (Step 000000): Train loss 9.980, Test loss 10.076
Ep 1 (Step 000005): Train loss 8.169, Test loss 8.493
Ep 1 (Step 000010): Train loss 6.793, Test loss 7.322
Epoch: 2
Ep 2 (Step 000015): Train loss 6.154, Test loss 6.950
Ep 2 (Step 000020): Train loss 5.885, Test loss 6.841
Ep 2 (Step 000025): Train loss 5.655, Test loss 6.852
Epoch: 3
Ep 3 (Step 000030): Train loss 5.313, Test loss 6.771
Ep 3 (Step 000035): Train loss 5.009, Test loss 6.746
Ep 3 (Step 000040): Train loss 5.153, Test loss 6.785
Epoch: 4
Ep 4 (Step 000045): Train loss 4.738, Test loss 6.604
Ep 4 (Step 000050): Train loss 4.212, Test loss 6.608
Ep 4 (Step 000055): Train loss 3.854, Test loss 6.589
Epoch: 5
Ep 5 (Step 000060): Train loss 3.265, Test loss 6.510
Ep 5 (Step 000065): Train loss 2.955, Test loss 6.524
Ep 5 (Step 000070): Train loss 2.383, Test loss 6.536
Epoch: 6
Ep 6 (Step 000075): Train loss 2.240, Test loss 6.565
Ep 6 (Step 000080): Train loss 1.799, Test loss 6.647
Ep 6 (Step 000085): Train l

Generate Text Function

In [21]:
def generate_text(model, idx, max_new_tokens, context_len, temperature=0.0, topk=None, eos_id=None):

    for _ in range(max_new_tokens):

        idx = idx.to(gpu)
        idx_cond = idx[:, -context_len:]
        

        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if topk is not None:
            top_logits, _ = torch.topk(logits, topk)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits/temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        
        else:
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        
        if eos_id == idx_next:
            break
        
        idx = torch.cat((idx, idx_next), dim=1)

    return idx



In [22]:
token_ids = generate_text(
    model = model,
    idx = text_to_token_ids("Every Effort Moves You", tokenizer),
    max_new_tokens=20,
    context_len=GPT_CONFIG_124M["context_length"],
    temperature=1.5,
    topk=50

)

print("Output Text: \n", token_ids_to_text(token_ids, tokenizer))

Output Text: 
 Every Effort Moves You lucky You him better; for he rose from the fact with equanimity.
She Arrt


Import Tensorflow

In [24]:
import tensorflow as tf
import tqdm

Download Pretrained Weights from OpenAI

In [25]:
from gpt_download3 import download_and_load_gpt2

In [26]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

/home/iztihad/Documents/GPT_2_from_scratch/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/checkpoint


/home/iztihad/Documents/GPT_2_from_scratch/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/encoder.json


/home/iztihad/Documents/GPT_2_from_scratch/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2/124M/model.ckpt.index


/home/iztihad/Documents/GPT_2_from_scratch/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/model.ckpt.meta


/home/iztihad/Documents/GPT_2_from_scratch/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [27]:
print(f"Settings: {settings}")
print(f"Params Dictionary Keys: {params.keys()}")

Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Params Dictionary Keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [28]:

NEW_CONFIG_GPT_2 = GPT_CONFIG_124M.copy()
NEW_CONFIG_GPT_2.update({"context_length": 1024, "qkv_bias": True})
print(NEW_CONFIG_GPT_2)

{'vocab_size': 50257, 'context_length': 1024, 'stride': 128, 'embedding_dim': 768, 'num_layers': 12, 'num_heads': 12, 'dropout': 0.1, 'qkv_bias': True}


In [29]:
gpt = GPTModel(NEW_CONFIG_GPT_2)
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformers): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm1): LayerNormaliztion()
      (norm2): LayerNormaliztion()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELUActivation()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
      

In [30]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformers): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm1): LayerNormaliztion()
      (norm2): LayerNormaliztion()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELUActivation()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
    

Updating the weights of our model with the downloaded weights from OpenAI

In [31]:
def assign(left, right):
    
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [32]:
print(params["blocks"][0])

{'attn': {'c_attn': {'b': array([ 0.48033914, -0.5254326 , -0.42926455, ...,  0.01257301,
       -0.04987717,  0.00324764], shape=(2304,), dtype=float32), 'w': array([[-0.4738484 , -0.26136586, -0.09780374, ...,  0.05132535,
        -0.0584389 ,  0.02499568],
       [ 0.08742206,  0.1473427 ,  0.23870145, ..., -0.05253514,
        -0.01125987, -0.01558759],
       [ 0.00388936,  0.06946629,  0.3668052 , ...,  0.11428114,
         0.03629516, -0.03184864],
       ...,
       [-0.25919554, -0.01636625,  0.19914557, ...,  0.00953369,
        -0.05159837,  0.03186192],
       [ 0.15165617,  0.2170211 ,  0.10434178, ...,  0.02933884,
        -0.04287174, -0.04746685],
       [-0.41001597, -0.19235404, -0.2400296 , ..., -0.00459218,
         0.00697855,  0.01984419]], shape=(768, 2304), dtype=float32)}, 'c_proj': {'b': array([ 1.50291592e-01, -1.54261470e-01, -1.46630690e-01, -9.91277322e-02,
        3.38026471e-02, -3.44475470e-02, -7.06353709e-02, -9.36073363e-02,
        8.11094940e-02,  

In [48]:
import numpy as np

def update_parameters(gpt, params):
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])

    for b in range(len(params["blocks"])):

        q_w, k_w, v_w = np.split((params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.transformers[b].attention.W_query.weight = assign(gpt.transformers[b].attention.W_query.weight, q_w.T)
        gpt.transformers[b].attention.W_key.weight = assign(gpt.transformers[b].attention.W_key.weight, k_w.T)
        gpt.transformers[b].attention.W_value.weight = assign(gpt.transformers[b].attention.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split((params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.transformers[b].attention.W_query.bias = assign(gpt.transformers[b].attention.W_query.bias, q_b)
        gpt.transformers[b].attention.W_key.bias = assign(gpt.transformers[b].attention.W_key.bias, k_b)
        gpt.transformers[b].attention.W_value.bias = assign(gpt.transformers[b].attention.W_value.bias, v_b)

        gpt.transformers[b].attention.out_proj.weight = assign(gpt.transformers[b].attention.out_proj.weight, 
                                                               params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.transformers[b].attention.out_proj.bias = assign(gpt.transformers[b].attention.out_proj.bias,
                                                                params["blocks"][b]["attn"]["c_proj"]["b"])
        
        gpt.transformers[b].ff.layers[0].weight = assign(gpt.transformers[b].ff.layers[0].weight,
                                                          params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.transformers[b].ff.layers[0].bias = assign(gpt.transformers[b].ff.layers[0].bias,
                                                          params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.transformers[b].ff.layers[2].weight = assign(gpt.transformers[b].ff.layers[2].weight,
                                                          params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.transformers[b].ff.layers[2].bias = assign(gpt.transformers[b].ff.layers[2].bias,
                                                          params["blocks"][b]["mlp"]["c_proj"]["b"])
        
        gpt.transformers[b].norm1.scale = assign(gpt.transformers[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.transformers[b].norm1.shift = assign(gpt.transformers[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.transformers[b].norm2.scale = assign(gpt.transformers[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.transformers[b].norm2.shift = assign(gpt.transformers[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])

    gpt.final_output.weight = assign(gpt.final_output.weight, params["wte"])

    return gpt
        
        



In [49]:
gpt = update_parameters(gpt, params)
gpt.to(gpu)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformers): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm1): LayerNormaliztion()
      (norm2): LayerNormaliztion()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELUActivation()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
      

In [65]:
torch.manual_seed(42)
token_ids = generate_text(
    model = model,
    idx = text_to_token_ids("Every Effort Moves You", tokenizer),
    max_new_tokens=28,
    context_len=GPT_CONFIG_124M["context_length"],
    temperature=1.5,
    topk=50

)

print("Output Text: \n", token_ids_to_text(token_ids, tokenizer))

Output Text: 
 Every Effort Moves You for you Stroud rich; and it was arm-room his last word. For Jack Gisburn! Poor women had dropped himself,


In [66]:
torch.manual_seed(42)
token_ids = generate_text(
    model = gpt,
    idx = text_to_token_ids("Every Effort Moves You", tokenizer),
    max_new_tokens=28,
    context_len=NEW_CONFIG_GPT_2["context_length"],
    temperature=1.5,
    topk=50

)

print("Output Text: \n", token_ids_to_text(token_ids, tokenizer))

Output Text: 
 Every Effort Moves You To the Table or the Mainstage, and a Better Match Ends Up All Alone For No Good Reason

What if all your efforts failed


Fine Tuning for Spam Classification

Importing, Balancing and Splitting the Dataset

In [7]:
import pandas as pd

spam_df = pd.read_csv("SMSSpamCollection", sep='\t', names=["Label", "Text"])
spam_df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [17]:
def create_balanced_dataset(df):

    num_spam = df[df["Label"] == "spam"].shape[0]

    ham_shuffle = df[df["Label"] == "ham"].sample(num_spam, random_state = 123)

    balanced_df = pd.concat([ham_shuffle, df[df["Label"] == "spam"]])

    return balanced_df

In [28]:
balanced_df = create_balanced_dataset(spam_df)
balanced_df["Label"] = balanced_df["Label"].map({"spam":1, "ham":0})
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [25]:
def random_split_dataset(df, train_ratio, val_ratio):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(train_ratio*len(df))
    val_end = train_end + int(val_ratio*len(df))

    train_df = df[:train_end]
    val_df = df[train_end:val_end]
    test_df = df[val_end:]

    return train_df, val_df, test_df

In [26]:
train_df, val_df, test_df = random_split_dataset(balanced_df, 0.7, 0.1)
print(len(train_df))
print(len(val_df))
print(len(test_df))

1045
149
300
